# 03_2b. XGBoost and LightGBM comparators / Сравнительные модели XGBoost и LightGBM

**EN.** Is "CatBoost is the main model" a choice of the method or of the library? XGBoost and LightGBM are tuned and fitted with exactly the protocol of M1: the 18 selected features, 40 Optuna trials, metric = ω-weighted log-loss on validation half A, number of trees by early stopping, 5 GroupKFold models whose logits are averaged, Platt calibration on half B. They run locally on the CPU (`scripts/tune.py xgboost|lightgbm y1 main`, `scripts/fit_models.py xgb_y1 lgb_y1`). Here: the tuning results, the validation predictions (needed for their calibrators on test) and the comparison with M1 on half A. On test they are reported as comparators next to B0 / M1 / M2; they do not enter any hypothesis rule.

**RU.** «Основная модель — CatBoost» — это выбор метода или библиотеки? XGBoost и LightGBM подбираются и обучаются точно по протоколу M1: 18 отобранных признаков, 40 попыток Optuna, метрика — log-loss с весом ω на половине A validation, число деревьев — по ранней остановке, 5 моделей GroupKFold с усреднением логитов, калибровка Платта на половине B. Считаются локально на процессоре (`scripts/tune.py xgboost|lightgbm y1 main`, `scripts/fit_models.py xgb_y1 lgb_y1`). Здесь: результаты подбора, прогнозы validation (нужны для их калибраторов на test) и сравнение с M1 на половине A. На test они приводятся как сравнительные модели рядом с B0 / M1 / M2 и не входят ни в одно решающее правило гипотез.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import polars as pl
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from xai_ews.config import feature_config
from xai_ews.evaluation.compare import pairwise
from xai_ews.evaluation.el import alert_mask
from xai_ews.evaluation.metrics import all_metrics
from xai_ews.models import registry
from xai_ews.models.calibration import Platt
from xai_ews.models.data import load_eval
from xai_ews.paths import MODELING_DIR, PREDICTIONS_DIR

SEED, B = 20261025, 1000
SEL = feature_config()["selected"]
FEATS, CATS = SEL["features"], SEL["categorical"]
MODELS = ["m1_y1", "xgb_y1", "lgb_y1"]

### 1. Tuning results / Результаты подбора

**EN.** Best trial of each library, its parameters, trees by early stopping, log-loss and AUC on half A, and the number of the 40 trials within 1 % of the best log-loss. The CatBoost row is M1 (from `tuning_catboost_summary.csv`).

**RU.** Лучшая попытка каждой библиотеки, её параметры, число деревьев по ранней остановке, log-loss и AUC на половине A и число из 40 попыток в пределах 1 % от лучшего log-loss. Строка CatBoost — это M1 (из `tuning_catboost_summary.csv`).

In [2]:
tuning = []
for lib in ("xgboost", "lightgbm"):
    name = f"{lib}_y1_main"
    st, df = registry.study(name), registry.trials_frame(name)
    df = df[df["state"] == "COMPLETE"]
    df.to_csv(MODELING_DIR / f"tuning_{name}.csv", index=False)
    tuning.append({"model": lib, "trials": len(df), "log_loss_A": st.best_value, "auc_A": st.best_trial.user_attrs["auc_A"],
                   "trees": st.best_trial.user_attrs["trees"], **st.best_params,
                   "within_1pct": int((df["value"] <= st.best_value * 1.01).sum())})
c = pd.read_csv(MODELING_DIR / "tuning_catboost_summary.csv").set_index("study").loc["catboost_y1_main"]
tuning.append({"model": "catboost (M1)", "trials": c["trials"], "log_loss_A": c["log_loss_A"], "auc_A": c["auc_A"],
               "trees": c["trees"], "depth": c["depth"], "learning_rate": c["learning_rate"], "l2": c["l2_leaf_reg"],
               "min_leaf": c["min_data_in_leaf"], "subsample": c["subsample"], "within_1pct": c["within_1pct"]})
tuning = pd.DataFrame(tuning).set_index("model")
tuning.to_csv(MODELING_DIR / "tuning_gbdt_summary.csv")
tuning

,trials,log_loss_A,auc_A,trees,depth,learning_rate,l2,min_leaf,subsample,within_1pct
model,,,,,,,,,,
xgboost,40.0000,0.0122,0.8633,713.0000,7.0000,0.0419,1.2756,619.0000,0.7522,39.0000
lightgbm,40.0000,0.0122,0.8631,326.0000,6.0000,0.0257,12.2745,13.0000,0.5671,38.0000
catboost (M1),40.0000,0.0122,0.8632,321.0000,4.0000,0.0630,3.3148,31.0000,0.9278,32.0000


**EN. How to read.** The three libraries end at the same place: half-A log-loss 0.012205 (XGBoost), 0.012217 (LightGBM) and 0.012236 (CatBoost) — within 0.25 % of each other — and AUC 0.863 for all three. XGBoost and LightGBM prefer deeper trees (depth 7 and 6) than CatBoost (4, the lower bound of its range); their optima are even flatter (39 and 38 of 40 trials within 1 % of the best, CatBoost 32), so the result does not hinge on the tuning.

**RU. Как читать.** Три библиотеки приходят к одному и тому же: log-loss на половине A 0,012205 (XGBoost), 0,012217 (LightGBM) и 0,012236 (CatBoost) — в пределах 0,25 % друг от друга — и AUC 0,863 у всех трёх. XGBoost и LightGBM выбирают более глубокие деревья (глубина 7 и 6), чем CatBoost (4, нижняя граница его диапазона); их оптимум ещё более плоский (39 и 38 из 40 попыток в пределах 1 % от лучшей, у CatBoost 32), так что результат не зависит от тонкостей подбора.

### 2. Validation: calibration and comparison with M1 / Validation: калибровка и сравнение с M1

**EN.** The validation logits are saved to `data/processed/predictions/<model>_validation.parquet` (the test notebooks read their Platt calibrators from there). Metrics on half A after Platt on half B; ΔAUC M1 − XGBoost and M1 − LightGBM with a cluster-bootstrap CI by loan and DeLong on one snapshot per loan (Holm over the two pairs); the alert list at calibrated PD ≥ 0.01.

**RU.** Логиты validation сохраняются в `data/processed/predictions/<model>_validation.parquet` (оттуда ноутбуки test берут калибраторы Платта). Метрики на половине A после Платта на половине B; ΔAUC M1 − XGBoost и M1 − LightGBM с кластерным bootstrap-интервалом по кредиту и DeLong на одном снимке на кредит (поправка Холма по двум парам); список тревог при откалиброванной PD ≥ 0,01.

In [3]:
preds = {}
for name in MODELS:
    f = PREDICTIONS_DIR / f"{name}_validation.parquet"
    if not f.exists():
        va = load_eval("validation", "y1", FEATS, CATS)
        va.select("loan_identifier", "m", "y1", "half", "upb", "omega").with_columns(
            pl.Series("logit", registry.predict(name, va))).write_parquet(f)
        del va; gc.collect()
    preds[name] = pl.read_parquet(f).sort("loan_identifier", "m")
keys = preds["m1_y1"].select("loan_identifier", "m", "y1", "half")
assert all(p.select("loan_identifier", "m").equals(keys.select("loan_identifier", "m")) for p in preds.values())
a, b = (keys["half"] == "A").to_numpy(), (keys["half"] == "B").to_numpy()
y = keys["y1"].to_numpy()
z = {n: p["logit"].to_numpy() for n, p in preds.items()}
rows = []
for n in MODELS:
    cal = Platt().fit(z[n][b], y[b])
    pa = cal.predict(z[n][a])
    al = alert_mask(pa)
    rows.append({"model": n, **all_metrics(y[a], pa), "alert_share": al.mean(), "alert_precision": y[a][al].mean(),
                 "alert_recall": y[a][al].sum() / y[a].sum()})
val = pd.DataFrame(rows).set_index("model")
val.to_csv(MODELING_DIR / "gbdt_validation_metrics.csv")
pw = pairwise(y[a], {n: z[n][a] for n in MODELS}, keys["loan_identifier"].to_numpy()[a],
              [("m1_y1", "xgb_y1"), ("m1_y1", "lgb_y1")], b=B, seed=SEED)
pw.to_csv(MODELING_DIR / "gbdt_validation_pairwise.csv", index=False)
display(val[["n", "events", "auc", "ks", "pr_auc", "brier", "log_loss", "cal_slope", "cal_intercept",
             "alert_share", "alert_precision", "alert_recall"]])
pw

,n,events,auc,ks,pr_auc,brier,log_loss,cal_slope,cal_intercept,alert_share,alert_precision,alert_recall
model,,,,,,,,,,,,
m1_y1,853029,1695,0.8629,0.5668,0.0156,0.0020,0.0123,1.0503,-0.0077,0.0357,0.0177,0.3186
xgb_y1,853029,1695,0.8627,0.5743,0.0158,0.0020,0.0123,1.0466,-0.0094,0.0397,0.0180,0.3599
lgb_y1,853029,1695,0.8625,0.5701,0.0161,0.0020,0.0123,1.0436,-0.0098,0.0374,0.0175,0.3304


,pair,delta_auc,lo,hi,share_le_0,delong_delta_one_snapshot,delong_p,one_snapshot_rows,one_snapshot_events,delong_p_holm
0,m1_y1 − xgb_y1,0.0003,-0.0023,0.0027,0.4040,0.0001,0.9153,287102,808,1.0000
1,m1_y1 − lgb_y1,0.0004,-0.0021,0.0029,0.3770,-0.0002,0.8747,287102,808,1.0000


**EN. How to read.** After Platt on half B the three models are indistinguishable on half A: AUC 0.8629 (M1), 0.8627 (XGBoost), 0.8625 (LightGBM); ΔAUC M1 − XGBoost +0.0003 [−0.0023; +0.0027], M1 − LightGBM +0.0004 [−0.0021; +0.0029], DeLong p 0.92 and 0.87 (Holm 1.0). Calibration is the same (slope 1.04–1.05, intercept −0.008…−0.010). At PD ≥ 0.01 XGBoost puts slightly more rows on the alert list (4.0 % vs 3.6 % for M1) and therefore catches more events (36 % vs 32 %) at the same precision (1.8 %) — a difference in how many alerts are raised, not in ranking quality. **Conclusion:** the choice of CatBoost as M1 is a choice of the method (gradient-boosted trees), not of the library; the comparators enter the test table descriptively.

**RU. Как читать.** После Платта на половине B три модели на половине A неразличимы: AUC 0,8629 (M1), 0,8627 (XGBoost), 0,8625 (LightGBM); ΔAUC M1 − XGBoost +0,0003 [−0,0023; +0,0027], M1 − LightGBM +0,0004 [−0,0021; +0,0029], p DeLong 0,92 и 0,87 (Холм — 1,0). Калибровка одинаковая (slope 1,04–1,05, intercept −0,008…−0,010). При PD ≥ 0,01 XGBoost ставит под тревогу чуть больше строк (4,0 % против 3,6 % у M1) и поэтому ловит больше событий (36 % против 32 %) при той же точности (1,8 %) — это разница в числе тревог, а не в качестве ранжирования. **Вывод:** выбор CatBoost как M1 — это выбор метода (градиентный бустинг деревьев), а не библиотеки; сравнительные модели входят в таблицу test описательно.